# 02 · Train six models × 3 seeds and measure energy

Models span ~75× in compute: ResNet-50, ResNet-18, EfficientNet-B0, MobileNetV3-Small, ViT-Tiny (all ImageNet-pretrained, adapted to 1-channel input) and a 0.3M-parameter SmallCNN trained from scratch.

Identical protocol for all: AdamW, cosine schedule, batch 64, ≤40 epochs, early stopping on validation loss (patience 6), flips + 90° rotations as augmentation.

**Energy:** measured with Zeus (`zeus-apple-silicon`), which reads the M3 Pro's on-chip energy counters for CPU, GPU, DRAM and ANE without admin rights. The `*_energy_mode` column records the backend used. We also record the hardware-independent cost: parameters, forward GFLOPs and wall time.

In [1]:
import sys, warnings; from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src')); warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, json
%matplotlib inline

In [2]:
from energy import zeus_available, measure_idle, PowerMeter
print('Zeus hardware energy counters available:', zeus_available())
print('idle package power (W):', round(measure_idle(15), 2))

Zeus hardware energy counters available: True
idle package power (W): 1.32


In [3]:
from data import make_splits, load_metadata, GasMapDataset
from models import MODELS, PRETTY
import train
df = pd.read_csv(ROOT/'data/processed/splits.csv')
tr, va, te = [GasMapDataset(df[df.split==s], augment=(s=='train')) for s in ['train','val','test']]
print(len(tr), len(va), len(te), 'device:', train.DEVICE)

1390 297 297 device: mps


In [4]:
SEEDS = [0, 1, 2]
rows = []
for name in MODELS:
    for seed in SEEDS:
        r = train.train_one(name, seed, tr, va, te)
        rows.append({k: v for k, v in r.items() if k not in ('test_keys', 'test_probs')})
        print(f"{PRETTY[name]:16s} s{seed}  acc={r['accuracy']:.3f}  auc={r['roc_auc']:.3f}  train={r['train_seconds']:.0f}s / {r['train_joules']:.0f} J  [{r['train_energy_mode']}]")
res = pd.DataFrame(rows); res.to_csv(ROOT/'results/runs.csv', index=False)

ResNet-50        s0  acc=0.902  auc=0.966  train=134s / 3118 J  [zeus_apple_silicon]
ResNet-50        s1  acc=0.899  auc=0.959  train=78s / 1810 J  [zeus_apple_silicon]
ResNet-50        s2  acc=0.936  auc=0.972  train=122s / 2823 J  [zeus_apple_silicon]
ResNet-18        s0  acc=0.906  auc=0.973  train=30s / 700 J  [zeus_apple_silicon]
ResNet-18        s1  acc=0.926  auc=0.968  train=30s / 708 J  [zeus_apple_silicon]
ResNet-18        s2  acc=0.939  auc=0.971  train=46s / 1081 J  [zeus_apple_silicon]
EfficientNet-B0  s0  acc=0.916  auc=0.950  train=90s / 1712 J  [zeus_apple_silicon]
EfficientNet-B0  s1  acc=0.916  auc=0.956  train=94s / 1778 J  [zeus_apple_silicon]
EfficientNet-B0  s2  acc=0.902  auc=0.945  train=79s / 1502 J  [zeus_apple_silicon]
MobileNetV3-S    s0  acc=0.892  auc=0.950  train=12s / 205 J  [zeus_apple_silicon]
MobileNetV3-S    s1  acc=0.906  auc=0.954  train=23s / 404 J  [zeus_apple_silicon]
MobileNetV3-S    s2  acc=0.916  auc=0.955  train=32s / 575 J  [zeus_apple_sili

In [5]:
cols = ['accuracy','f1','recall','roc_auc','ece','params','gflops','epochs','train_seconds','train_joules','train_kgco2e','inf_per1k_joules','inf_per1k_seconds']
summary = res.groupby('model')[cols].agg(['mean','std'])
summary.round(4)

accuracy              f1          recall          \
                          mean     std    mean     std    mean     std   
model                                                                    
efficientnet_b0         0.9113  0.0078  0.9100  0.0096  0.9009  0.0256   
mobilenetv3_small_100   0.9046  0.0118  0.9055  0.0107  0.9167  0.0078   
resnet18                0.9237  0.0169  0.9250  0.0160  0.9437  0.0141   
resnet50                0.9125  0.0205  0.9116  0.0216  0.9077  0.0384   
small_cnn               0.8103  0.0152  0.7983  0.0268  0.7590  0.0900   
vit_tiny_patch16_224    0.9192  0.0034  0.9204  0.0031  0.9369  0.0078   

                      roc_auc             ece          ... train_seconds  \
                         mean     std    mean     std  ...          mean   
model                                                  ...                 
efficientnet_b0        0.9500  0.0055  0.0597  0.0076  ...       88.0249   
mobilenetv3_small_100  0.9534  0.0026  0.0633  0.0048  ...       22.3744   
resnet18               0.9706  0.0028  0.0398  0.0049  ...       35.5734   
resnet50               0.9658  0.0062  0.0379  0.0129  ...      111.1370   
small_cnn              0.9042  0.0198  0.0392  0.0086  ...       23.6985   
vit_tiny_patch16_224   0.9614  0.0108  0.0368  0.0071  ...       30.0686   

                               train_joules           train_kgco2e       \
                           std         mean       std         mean  std   
model                                                                     
efficientnet_b0         7.7205    1664.1660  144.1779       0.0001  0.0   
mobilenetv3_small_100   9.9231     394.5313  185.1954       0.0000  0.0   
resnet18                9.2552     829.9733  217.7364       0.0000  0.0   
resnet50               29.5135    2584.0313  686.0288       0.0001  0.0   
small_cnn               5.3138     517.4233  113.8864       0.0000  0.0   
vit_tiny_patch16_224    2.8169     723.6533   70.5760       0.0000  0.0   

                      inf_per1k_joules         inf_per1k_seconds          
                                  mean     std              mean     std  
model                                                                     
efficientnet_b0                19.9995  0.2160            0.9565  0.0007  
mobilenetv3_small_100           4.2603  0.0345            0.2238  0.0004  
resnet18                        8.7782  0.0785            0.3721  0.0017  
resnet50                       31.2628  1.4551            1.3006  0.0213  
small_cnn                       5.7348  0.0136            0.2659  0.0003  
vit_tiny_patch16_224            7.4612  0.0715            0.3326  0.0010  

[6 rows x 26 columns]